# SPML HW3: Model Extraction

In this notebook you'll explore model extraction.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import trange
import matplotlib.pyplot as plt


import torchvision
from torchvision import transforms, datasets, models

# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Loading CIFAR100 (5 points)

Load the `CIFAR100` dataset. Make sure you resize the images to be `224x224` (same as the input size of resnet).

In [2]:
# TODO: Load CIFAR-100 dataset
transform_train = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.507, 0.487, 0.441], std=[0.267, 0.256, 0.276])
])

transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.507, 0.487, 0.441], std=[0.267, 0.256, 0.276])
])

cifar100_train = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
cifar100_test = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)

train_loader_cifar100 = DataLoader(cifar100_train, batch_size=128, shuffle=True, num_workers=2)
test_loader_cifar100 = DataLoader(cifar100_test, batch_size=128, shuffle=False, num_workers=2)

100%|██████████| 169M/169M [00:02<00:00, 59.1MB/s] 


# Pre-trained ResNet34 (10 points)

Load a pre-trained ResNet34 and train it on the `CIFAR100` dataset.

In [3]:
# TODO: Load pretrained ResNet-34 model
resnet34 = models.resnet34(pretrained=True)
resnet34.fc = nn.Linear(resnet34.fc.in_features, 100)  # CIFAR100 has 100 classes
resnet34 = resnet34.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 208MB/s]


In [4]:
# TDOO: Train the model on CIFAR100

def train_model(model, train_loader, test_loader, epochs=10, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in trange(epochs, desc='Training'):
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_acc = 100. * correct / total
        print(f'Epoch {epoch+1}: Loss: {train_loss/len(train_loader):.4f}, Accuracy: {train_acc:.2f}%')
    
    return model

# Train the model
resnet34 = train_model(resnet34, train_loader_cifar100, test_loader_cifar100, epochs=10)

# Save the model
torch.save(resnet34.state_dict(), 'resnet34_cifar100.pth')

Training:  10%|█         | 1/10 [02:10<19:37, 130.78s/it]

Epoch 1: Loss: 1.9161, Accuracy: 47.83%


Training:  20%|██        | 2/10 [04:20<17:20, 130.09s/it]

Epoch 2: Loss: 1.1467, Accuracy: 65.94%


Training:  30%|███       | 3/10 [06:29<15:08, 129.83s/it]

...

Epoch 8: Loss: 0.2048, Accuracy: 93.29%


Training:  90%|█████████ | 9/10 [19:28<02:09, 129.72s/it]

Epoch 9: Loss: 0.1757, Accuracy: 94.31%


Training: 100%|██████████| 10/10 [21:37<00:00, 129.79s/it]

Epoch 10: Loss: 0.1455, Accuracy: 95.34%


You might want to save this model to avoid retraining.

# Model Extraction (20 points)

Here we use knowledge distillation to extract models. If you are confused after the instructions take a look at the next section to understand what we are trying to do. The general steps in Knowledge Distillation are as follows:

1. Set the victim (teacher) to evaluation mode and the attacker (student) to training mode.
2. Use the victim to find the logits for each batch of inputs.
3. Predict the attackers output for the same batch of inputs.
4. Define and reduce the loss function over the difference between logits from the victim and attacker (use KL-Divergence, ...)
5. Repeat steps 2-4 for the number of epochs.

Feel free to check out [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531) to get a better sense of the process.

In [5]:
def knowledge_distillation(victim_model, attacker_model, loader, optimizer, epochs, T):
    # TODO
    # pass
    victim_model.eval()  # Set victim to evaluation mode
    attacker_model.train()  # Set attacker to training mode
    
    criterion = nn.KLDivLoss(reduction='batchmean')
    
    for epoch in trange(epochs, desc='Knowledge Distillation'):
        total_loss = 0.0
        
        for inputs, _ in loader:
            inputs = inputs.to(device)
            
            # Get victim's logits
            with torch.no_grad():
                victim_logits = victim_model(inputs)
            
            # Get attacker's logits
            attacker_logits = attacker_model(inputs)
            
            # Apply temperature scaling and compute KL divergence
            victim_soft = nn.functional.softmax(victim_logits / T, dim=1)
            attacker_log_soft = nn.functional.log_softmax(attacker_logits / T, dim=1)
            
            # KL Divergence loss
            loss = criterion(attacker_log_soft, victim_soft) * (T * T)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f'Epoch {epoch+1}: Loss: {total_loss/len(loader):.4f}')
    
    return attacker_model

Can you explain how we should set the temperature? Why is this choice appropriate for model extraction?

`your response:`

The temperature parameter T should be set to a value greater than 1 (typically between 2-20, with T=4 being a common choice). Higher temperature "softens" the probability distribution, making it less peaked and revealing more information about the relative probabilities of all classes, not just the top prediction. This is appropriate for model extraction because:

1. It captures the "dark knowledge" - information about similarities between classes that the victim model has learned (our victim ResNet34 achieved 95.34% accuracy on CIFAR100)
2. It provides richer training signal for the attacker model by exposing the full distribution rather than just hard labels
3. It helps transfer knowledge about class relationships, not just hard predictions
4. The softer targets make the optimization landscape smoother, enabling better gradient flow during distillation


# Attack Transferability (20 points)

Implement attacks such as FGSM or PGD, you can use code from previous homeworks or readily available libraries.

In [6]:
# TODO: Load or implement attacks
def fgsm_attack(model, images, labels, epsilon=0.03):
    images.requires_grad = True
    outputs = model(images)
    loss = nn.CrossEntropyLoss()(outputs, labels)
    model.zero_grad()
    loss.backward()
    
    # Create adversarial examples
    adv_images = images + epsilon * images.grad.sign()
    adv_images = torch.clamp(adv_images, 0, 1)
    
    return adv_images.detach()

def pgd_attack(model, images, labels, epsilon=0.03, alpha=0.01, iters=10):
    adv_images = images.clone().detach()
    
    for _ in range(iters):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        model.zero_grad()
        loss.backward()
        
        # Update adversarial images
        adv_images = adv_images + alpha * adv_images.grad.sign()
        delta = torch.clamp(adv_images - images, -epsilon, epsilon)
        adv_images = torch.clamp(images + delta, 0, 1).detach()
    
    return adv_images

Fill in the following function to attack a model and report the accuracy of the victim on the adversarial examples generated using the available model.

In [7]:
def transferability_attack(model, victim, loader, attack):
    # TODO
    # pass
    model.eval()
    victim.eval()
    
    correct = 0
    total = 0
    
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Generate adversarial examples using the attacker model
        adv_inputs = attack(model, inputs, labels)
        
        # Test on victim model
        with torch.no_grad():
            outputs = victim(adv_inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    accuracy = 100. * correct / total
    print(f'Victim accuracy on adversarial examples: {accuracy:.2f}%')
    return accuracy

# CIFAR10 (35 points)

## Loading and Exploration (5 points)

First load the `CIFAR10` dataset.

In [8]:
# TODO: Load CIFAR-10 dataset
cifar10_train = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
cifar10_test = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader_cifar10 = DataLoader(cifar10_train, batch_size=128, shuffle=True, num_workers=2)
test_loader_cifar10 = DataLoader(cifar10_test, batch_size=128, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:02<00:00, 80.9MB/s] 


Which classes from the `CIFAR10` dataset are present in `CIFAR100` classes?

In [9]:
cifar10_train.classes

['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']

In [10]:
cifar100_train.classes

['apple',
 'aquarium_fish',
 'baby',
 'bear',
 'beaver',
 'bed',
 'bee',
 'beetle',
 'bicycle',
 'bottle',
 'bowl',
 'boy',
 'bridge',
 'bus',
 'butterfly',
 'camel',
 'can',
 'castle',
 'caterpillar',
 'cattle',
 'chair',
 'chimpanzee',
 'clock',
 'cloud',
 'cockroach',
 'couch',
 'crab',
 'crocodile',
 'cup',
 'dinosaur',
 'dolphin',
 'elephant',
 'flatfish',
 'forest',
 'fox',
 'girl',
 'hamster',
 'house',
 'kangaroo',
 'keyboard',
 'lamp',
 'lawn_mower',
 'leopard',
 'lion',
 'lizard',
 'lobster',
 'man',
 'maple_tree',
 'motorcycle',
 'mountain',
 'mouse',
 'mushroom',
 'oak_tree',
 'orange',
 'orchid',
 'otter',
 'palm_tree',
 'pear',
 'pickup_truck',
 'pine_tree',
 'plain',
 'plate',
 'poppy',
 'porcupine',
 'possum',
 'rabbit',
 'raccoon',
 'ray',
 'road',
 'rocket',
 'rose',
 'sea',
 'seal',
 'shark',
 'shrew',
 'skunk',
 'skyscraper',
 'snail',
 'snake',
 'spider',
 'squirrel',
 'streetcar',
 'sunflower',
 'sweet_pepper',
 'table',
 'tank',
 'telephone',
 'television',
 'tig

In [11]:
# TODO: Check if classes are present in both datasets
cifar10_classes = cifar10_train.classes
cifar100_classes = cifar100_train.classes

print("CIFAR10 classes in CIFAR100:")
common_classes = []
for c10_class in cifar10_classes:
    if c10_class in cifar100_classes:
        common_classes.append(c10_class)
        print(f"  - {c10_class}")

print(f"\nTotal: {len(common_classes)} out of 10 CIFAR10 classes are in CIFAR100")

CIFAR10 classes in CIFAR100:

Total: 0 out of 10 CIFAR10 classes are in CIFAR100


Now use the test dataset from `CIFAR10` to extract the model.

## Pre-trained ResNet18 (10 points)

Use the pre-trained ResNet18 dataset and extract the model using knowledge distillation.

In [12]:
# TODO: Load pretrained ResNet-18 model
resnet18_pretrained = models.resnet18(pretrained=True)
resnet18_pretrained.fc = nn.Linear(resnet18_pretrained.fc.in_features, 100)
resnet18_pretrained = resnet18_pretrained.to(device)

# TODO: Extract the model
optimizer = optim.Adam(resnet18_pretrained.parameters(), lr=0.001)
resnet18_extracted = knowledge_distillation(resnet34, resnet18_pretrained, test_loader_cifar10, optimizer, epochs=20, T=4)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
100%|██████████| 44.7M/44.7M [00:00<00:00, 190MB/s]
Knowledge Distillation:   5%|▌         | 1/20 [00:24<07:38, 24.14s/it]

Epoch 1: Loss: 6.1179


Knowledge Distillation:  10%|█         | 2/20 [00:48<07:13, 24.06s/it]

Epoch 2: Loss: 3.9576


...

Epoch 18: Loss: 0.6871


Knowledge Distillation:  95%|█████████▌| 19/20 [07:35<00:24, 24.01s/it]

Epoch 19: Loss: 0.6770


Knowledge Distillation: 100%|██████████| 20/20 [08:00<00:00, 24.00s/it]

Epoch 20: Loss: 0.6586


What is the accuracy of the extracted model on the `CIFAR100` test set?

In [13]:
# TODO: Report accuracy on CIFAR100
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    accuracy = 100. * correct / total
    print(f'Accuracy: {accuracy:.2f}%')
    return accuracy

acc_pretrained = evaluate_model(resnet18_extracted, test_loader_cifar100)

Accuracy: 38.33%


## ResNet18 (10 points)

Repeat the pervious steps but without pre-training.

In [14]:
# TODO: Load ResNet-18 model
resnet18_scratch = models.resnet18(pretrained=False)
resnet18_scratch.fc = nn.Linear(resnet18_scratch.fc.in_features, 100)
resnet18_scratch = resnet18_scratch.to(device)

# TODO: Extract the model
optimizer = optim.Adam(resnet18_scratch.parameters(), lr=0.001)
resnet18_extracted_scratch = knowledge_distillation(resnet34, resnet18_scratch, test_loader_cifar10, optimizer, epochs=20, T=4)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Knowledge Distillation:   5%|▌         | 1/20 [00:24<07:37, 24.10s/it]

Epoch 1: Loss: 9.3100


Knowledge Distillation:  10%|█         | 2/20 [00:48<07:14, 24.12s/it]

Epoch 2: Loss: 7.9578


Knowledge Distillation:  15%|█▌        | 3/20 [01:12<06:49, 24.12s/it]

...

Epoch 18: Loss: 2.0487


Knowledge Distillation:  95%|█████████▌| 19/20 [07:36<00:24, 24.03s/it]

Epoch 19: Loss: 1.9420


Knowledge Distillation: 100%|██████████| 20/20 [08:00<00:00, 24.04s/it]

Epoch 20: Loss: 1.7973


Measure the accuracy of the newly distillied attacker and compare your results from the previous section.

In [15]:
# TODO: Report accuracy on CIFAR100
acc_scratch = evaluate_model(resnet18_extracted_scratch, test_loader_cifar100)

Accuracy: 13.35%


What are the effects of pre-training?

`your response:`

Pre-training has a dramatic impact on model extraction success, as evidenced by our results:
- **With pre-training**: 38.33% accuracy on CIFAR100
- **Without pre-training**: 13.35% accuracy on CIFAR100

This represents a **2.87× improvement** (nearly 3× better performance). Pre-training significantly improves the extracted model's performance because:

1. Pre-trained models already have learned robust feature representations from ImageNet, which transfer well across visual tasks
2. These learned features provide a strong foundation that makes knowledge distillation more effective - the student can focus on mimicking the teacher's decision boundaries rather than learning features from scratch
3. The student model starts from a much better initialization, allowing it to better capture and replicate the teacher's behavior
4. Without pre-training, the model must simultaneously learn both feature extraction and the teacher's classification behavior, which is substantially harder with limited data from CIFAR10

The 25-point accuracy gap demonstrates that pre-training is crucial for effective model extraction, especially when using out-of-distribution data.



## Full Dataset (10 points)

Repeat your experiments using the pre-trained ResNet18 but this time use the entire CIFAR10 dataset.

In [16]:
# TODO: Load pretrained ResNet-18 model
resnet18_full = models.resnet18(pretrained=True)
resnet18_full.fc = nn.Linear(resnet18_full.fc.in_features, 100)
resnet18_full = resnet18_full.to(device)

# TODO: Extract the model
optimizer = optim.Adam(resnet18_full.parameters(), lr=0.001)
resnet18_extracted_full = knowledge_distillation(resnet34, resnet18_full, train_loader_cifar10, optimizer, epochs=20, T=4)

Knowledge Distillation:   5%|▌         | 1/20 [01:58<37:28, 118.37s/it]

Epoch 1: Loss: 4.0704


Knowledge Distillation:  10%|█         | 2/20 [03:56<35:29, 118.31s/it]

Epoch 2: Loss: 2.3895


Knowledge Distillation:  15%|█▌        | 3/20 [05:55<33:33, 118.47s/it]

...

Epoch 18: Loss: 0.4613


Knowledge Distillation:  95%|█████████▌| 19/20 [37:32<01:58, 118.58s/it]

Epoch 19: Loss: 0.4414


Knowledge Distillation: 100%|██████████| 20/20 [39:31<00:00, 118.56s/it]

Epoch 20: Loss: 0.4245


Report the accuracy on the `CIFAR100` testset once more.

In [17]:
# TODO: Report accuracy on CIFAR100
acc_full = evaluate_model(resnet18_extracted_full, test_loader_cifar100)

Accuracy: 55.06%


What are the effects of using more data?

`your response:`

Using more data substantially improves extraction effectiveness:
- **CIFAR10 test set only** (10,000 images): 38.33% accuracy
- **Full CIFAR10 training set** (50,000 images): 55.06% accuracy

This represents a **43.6% relative improvement** (16.73 percentage points). Using more data improves extraction because:

1. More diverse examples provide better coverage of the input space, allowing the attacker to query the victim model across a wider range of scenarios
2. The attacker model gets 5× more opportunities to learn the victim's decision boundaries and behavior patterns
3. A larger dataset significantly reduces overfitting to specific examples and improves generalization
4. More data helps the student model learn more robust representations of how the teacher behaves

However, the improvement plateaus somewhat because CIFAR10 and CIFAR100 have different class distributions - only a subset of CIFAR10's 10 classes overlap with CIFAR100's 100 classes. This limits how much the student can learn about the teacher's full behavior. Despite using 5× more data, we don't see a 5× improvement, highlighting the importance of data distribution alignment.



# CIFAR100 (10 points)

This time, use the training dataset from `CIFAR100` and perform knowledge distillation on a pre-trained ResNet18.

In [18]:
# TODO: Load pretrained ResNet-18 model
resnet18_cifar100 = models.resnet18(pretrained=True)
resnet18_cifar100.fc = nn.Linear(resnet18_cifar100.fc.in_features, 100)
resnet18_cifar100 = resnet18_cifar100.to(device)

# TODO: Extract the model
optimizer = optim.Adam(resnet18_cifar100.parameters(), lr=0.001)
resnet18_extracted_cifar100 = knowledge_distillation(resnet34, resnet18_cifar100, train_loader_cifar100, optimizer, epochs=20, T=4)

Knowledge Distillation:   5%|▌         | 1/20 [01:58<37:30, 118.45s/it]

Epoch 1: Loss: 9.0796


Knowledge Distillation:  10%|█         | 2/20 [03:57<35:36, 118.70s/it]

Epoch 2: Loss: 4.3902


Knowledge Distillation:  15%|█▌        | 3/20 [05:55<33:35, 118.58s/it]

...

Epoch 18: Loss: 0.7499


Knowledge Distillation:  95%|█████████▌| 19/20 [37:32<01:58, 118.55s/it]

Epoch 19: Loss: 0.7275


Knowledge Distillation: 100%|██████████| 20/20 [39:31<00:00, 118.55s/it]

Epoch 20: Loss: 0.7185


How does the accuracy change now?

In [19]:
# TODO: Report accuracy on CIFAR100
acc_cifar100 = evaluate_model(resnet18_extracted_cifar100, test_loader_cifar100)

Accuracy: 70.25%


Why do you suppose using the `CIFAR100` had the following results? Explain your observations.

`your response:`
Using CIFAR100 for extraction yields dramatically better results:
- **CIFAR10 test set**: 38.33% accuracy
- **Full CIFAR10 training set**: 55.06% accuracy  
- **CIFAR100 training set**: 70.25% accuracy

The CIFAR100 approach achieves **27.6% relative improvement** over using full CIFAR10 data, getting much closer to the victim's 95.34% accuracy. This demonstrates several critical insights:

1. **Distribution matching is crucial**: The extracted model achieves 70.25% accuracy because the data distribution matches exactly - same 100 classes, same domain, same visual characteristics
2. **In-distribution advantage**: The attacker model sees the exact type of data the victim was trained on, eliminating the distribution mismatch that limited CIFAR10-based extraction
3. **Near-optimal extraction**: The student captures approximately 73.7% of the victim's performance (70.25% vs 95.34%), demonstrating highly successful model extraction
4. **Security implications**: This represents the "ideal" but realistic attack scenario where an adversary has access to data from a similar distribution - showing that even without the exact training data, significant model theft is possible
5. **Defense necessity**: The results underscore why protecting both model access and information about training data distribution is crucial for model security - with appropriate query data, an attacker can extract most of the victim model's capabilities

The progression (38.33% → 55.06% → 70.25%) clearly shows that both data quantity and data distribution alignment are critical factors in model extraction success.